# Lab 13: Dashboards, Streaming, and Deployment

This notebook documents and demonstrates the Lab 13 dashboard implementation.

The project uses:
- Dash for the dashboard interface
- Plotly for visualizations
- MongoDB for database storage
- Docker and Docker Compose for deployment
- `dcc.Interval` for simulated live dashboard updates

## 1. Dashboard Overview

The dashboard is an interactive web application built with Dash.

It displays analytics from the cleaned Geoapify location dataset. The original dataset contains fields such as:

- `name`
- `city`
- `country`
- `categories`
- `longitude`
- `latitude`
- `website`
- `timestamp_year`

For dashboard compatibility, the data is normalized into fields such as:

- `title`
- `genre`
- `year`
- `rating`
- `revenue`
- `budget`

In [1]:
from pathlib import Path

project_files = [
    "app.py",
    "Dockerfile",
    "docker-compose.yml",
    ".dockerignore",
    "src/dashboard/data_access.py",
    "src/dashboard/layout.py",
    "src/dashboard/callbacks.py",
    "scripts/seed_mongo.py",
]

for file in project_files:
    path = Path("..") / file
    print(file, "FOUND" if path.exists() else "MISSING")

app.py FOUND
Dockerfile FOUND
docker-compose.yml FOUND
.dockerignore FOUND
src/dashboard/data_access.py FOUND
src/dashboard/layout.py FOUND
src/dashboard/callbacks.py FOUND
scripts/seed_mongo.py FOUND


## 2. Data Source

The application first tries to read data from MongoDB.

If MongoDB is unavailable or empty, the application falls back to the cleaned CSV file.

The MongoDB collection used in this project is:

```text
Database: movie_analytics
Collection: movies
```

In [2]:
import sys
from pathlib import Path

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.dashboard.data_access import load_movies, get_genres, get_year_range, filter_movies

df = load_movies()
print("Rows:", len(df))
print("Columns:", df.columns.tolist())
df.head()

Rows: 771
Columns: ['source', 'timestamp', 'name', 'city', 'country', 'categories', 'longitude', 'latitude', 'website', 'formatted_address', 'timestamp_year', 'title', 'genre', 'year', 'rating', 'revenue', 'budget']


,source,timestamp,name,city,country,categories,longitude,latitude,website,formatted_address,timestamp_year,title,genre,year,rating,revenue,budget
0,geoapify_json,2026-03-24 00:17:00.235,dom oružanih snaga bih,sarajevo,bosnia and herzegovina,"activity, activity.community_center, building,...",18.424120,43.857283,http://www.mod.gov.ba/,"armed forces hall of bosnia and herzegovina, z...",2026,dom oružanih snaga bih,activity,2026,1,1,1
1,geoapify_json,2026-03-24 00:17:00.273,zajednički štab oružanih snaga bih,sarajevo,bosnia and herzegovina,"building, building.historic, building.military...",18.429970,43.856776,http://www.mod.gov.ba/,"joint staff of the armed forces of bih, bistri...",2026,zajednički štab oružanih snaga bih,building,2026,1,1,1
2,geoapify_json,2026-03-24 00:17:00.274,gazi husrev-begov bezistan,sarajevo,bosnia and herzegovina,"building, building.commercial, building.histor...",18.428079,43.858747,unknown,"gazi husrev bey's bezistan, mudželiti mali, 71...",2026,gazi husrev-begov bezistan,building,2026,1,1,1
3,geoapify_json,2026-03-24 00:17:00.274,aškenaška sinagoga,sarajevo,bosnia and herzegovina,"building, building.historic, building.place_of...",18.425081,43.856311,unknown,"ashkenazi synagogue, hamdije kreševljakovića, ...",2026,aškenaška sinagoga,building,2026,1,1,1
4,geoapify_json,2026-03-24 00:17:00.275,katedrala srca isusova,sarajevo,bosnia and herzegovina,"building, building.historic, building.place_of...",18.425363,43.859430,https://katedrala-sarajevo.com/,"sacred heart cathedral, trg fra grge martića, ...",2026,katedrala srca isusova,building,2026,1,1,1


## 3. MongoDB Seeding

MongoDB is seeded using the script:

```powershell
python scripts/seed_mongo.py
```

Expected result:

```text
Seeded 771 records into movie_analytics.movies
```

In [6]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017")
db = client["movie_analytics"]
collection = db["movies"]

count = collection.count_documents({})
print("MongoDB document count:", count)

sample = collection.find_one()
sample

MongoDB document count: 771


{'_id': '69c1c9ec21c9fc4ece6b369d',
 'source': 'geoapify_json',
 'timestamp': '2026-03-24 00:17:00.235',
 'name': 'dom oružanih snaga bih',
 'city': 'sarajevo',
 'country': 'bosnia and herzegovina',
 'categories': 'activity, activity.community_center, building, building.historic, building.public_and_civil, heritage, tourism, tourism.sights',
 'longitude': 18.424119991873905,
 'latitude': 43.857283450536826,
 'website': 'http://www.mod.gov.ba/',
 'formatted_address': 'armed forces hall of bosnia and herzegovina, zelenih beretki 2, 71000 sarajevo, bosnia and herzegovina',
 'timestamp_year': 2026,
 'title': 'dom oružanih snaga bih',
 'genre': 'activity',
 'year': 2026,
 'rating': 1,
 'revenue': 1,
 'budget': 1}

## 4. Dashboard Interactivity

The dashboard contains multiple interactive components:

- Genre/category dropdown
- Year range slider
- Search input
- Live ticker interval

All main charts respond to the same filters.

In [5]:
genres = get_genres()
year_min, year_max = get_year_range()

print("Available genres/categories:", genres[:10])
print("Year range:", year_min, "-", year_max)

filtered_df = filter_movies(
    genre="All",
    year_range=[year_min, year_max],
    search_text=""
)

print("Filtered rows:", len(filtered_df))
filtered_df.head()

Available genres/categories: ['All', 'access', 'activity', 'building', 'education', 'heritage', 'leisure', 'natural', 'populated_place', 'tourism']
Year range: 2026 - 2026
Filtered rows: 771


,source,timestamp,name,city,country,categories,longitude,latitude,website,formatted_address,timestamp_year,title,genre,year,rating,revenue,budget
0,geoapify_json,2026-03-24 00:17:00.235,dom oružanih snaga bih,sarajevo,bosnia and herzegovina,"activity, activity.community_center, building,...",18.424120,43.857283,http://www.mod.gov.ba/,"armed forces hall of bosnia and herzegovina, z...",2026,dom oružanih snaga bih,activity,2026,1,1,1
1,geoapify_json,2026-03-24 00:17:00.273,zajednički štab oružanih snaga bih,sarajevo,bosnia and herzegovina,"building, building.historic, building.military...",18.429970,43.856776,http://www.mod.gov.ba/,"joint staff of the armed forces of bih, bistri...",2026,zajednički štab oružanih snaga bih,building,2026,1,1,1
2,geoapify_json,2026-03-24 00:17:00.274,gazi husrev-begov bezistan,sarajevo,bosnia and herzegovina,"building, building.commercial, building.histor...",18.428079,43.858747,unknown,"gazi husrev bey's bezistan, mudželiti mali, 71...",2026,gazi husrev-begov bezistan,building,2026,1,1,1
3,geoapify_json,2026-03-24 00:17:00.274,aškenaška sinagoga,sarajevo,bosnia and herzegovina,"building, building.historic, building.place_of...",18.425081,43.856311,unknown,"ashkenazi synagogue, hamdije kreševljakovića, ...",2026,aškenaška sinagoga,building,2026,1,1,1
4,geoapify_json,2026-03-24 00:17:00.275,katedrala srca isusova,sarajevo,bosnia and herzegovina,"building, building.historic, building.place_of...",18.425363,43.859430,https://katedrala-sarajevo.com/,"sacred heart cathedral, trg fra grge martića, ...",2026,katedrala srca isusova,building,2026,1,1,1


## 5. Charts Implemented

The dashboard includes the following visualizations:

1. Top 10 places by count-style metric
2. Website availability / rating distribution
3. Budget vs revenue style scatter chart
4. Yearly trend chart
5. Simulated live ticker chart

The charts are implemented in:

```text
src/dashboard/callbacks.py
```

The layout is implemented in:

```text
src/dashboard/layout.py
```

## 6. Running the Dashboard Locally

To run the dashboard without Docker:

```powershell
python app.py
```

Then open:

```text
http://127.0.0.1:8050
```

## 7. Docker Deployment

### Prerequisites

Before deployment, make sure:

- Docker Desktop is installed
- Docker Desktop is running
- Project files are saved
- MongoDB seed script exists
- Dockerfile and docker-compose.yml exist

### Start the full stack

Run:

```powershell
docker compose up --build
```

This starts:

- Dash web app on port `8050`
- MongoDB on port `27017`

### Seed MongoDB

In another terminal, run:

```powershell
python scripts/seed_mongo.py
```

### Verify deployment

Open:

```text
http://localhost:8050
```

Expected result:

- Dashboard loads successfully
- KPI cards show 771 records
- Charts display real data
- Dropdown, slider, and search input work
- Live ticker updates automatically

## 8. Checking Containers

To confirm that both services are running:

```powershell
docker ps
```

Expected containers:

```text
movie_dashboard_web
movie_dashboard_db
```

The web container should expose:

```text
0.0.0.0:8050->8050/tcp
```

The MongoDB container should expose:

```text
0.0.0.0:27017->27017/tcp
```

## 9. Stopping the Stack

To stop the Docker deployment:

```powershell
docker compose down
```

This stops and removes the running containers.

## 10. Troubleshooting

### Problem: localhost refused to connect

Check containers:

```powershell
docker ps
```

If only MongoDB is running, check web logs:

```powershell
docker compose logs web
```

Then rebuild:

```powershell
docker compose down
docker compose up --build
```

### Problem: Dashboard shows zero values

This usually means the data columns do not match the expected dashboard columns.

Fix:

- Check MongoDB sample record
- Normalize fields in `data_access.py`
- Re-run `python scripts/seed_mongo.py`
- Rebuild Docker stack

### Problem: MongoDB collection is empty

Run:

```powershell
python scripts/seed_mongo.py
```

Then verify:

```powershell
python -c "from pymongo import MongoClient; c=MongoClient('mongodb://localhost:27017'); print(c.movie_analytics.movies.count_documents({}))"
```

## 11. Final Verification

The Lab 13 implementation is complete when:

- `docker ps` shows both `movie_dashboard_web` and `movie_dashboard_db`
- `http://localhost:8050` opens successfully
- Dashboard shows 771 records
- Filters affect all charts
- Live ticker updates automatically
- Notebook documents deployment commands and troubleshooting steps